# Typed domains and legal moves

`pypft.forward_pft`/`pypft.inverse_pft` (earlier notebooks) already do every
numerical step correctly. `pypft.domains` adds nothing numerical on top of
them -- it adds a *typed* way to name where a polar array sits along the
chain, and to walk between those points one verified step at a time:

```text
SPACE_POLAR --DFT--> SPACE_HARMONIC --DHT--> FREQUENCY_HARMONIC --IDFT--> FREQUENCY_POLAR
```

Each `pypft.Domain` member's first word (`SPACE`/`FREQUENCY`) is the radial
coordinate, changed only by the discrete Hankel transform (DHT); the second
word (`POLAR`/`HARMONIC`) is the angular coordinate, changed only by the
angular DFT/IDFT. `pypft.BaseSignal` and its four subclasses -- one per
`Domain` member -- wrap a `(values, grid)` pair with the domain it is
currently in, and know only the neighbouring domains they may legally step
to.


In [ ]:
import numpy as np

import pypft


## Wrapping a signal

Any array that could be passed to `forward_pft` can instead be wrapped in
`pypft.SpacePolarSignal` -- the domain PyPFT's own axis layout starts in,
`f(r, theta)`:


In [ ]:
grid = pypft.PolarGrid(n_radial=382, n_angular=15, R=40.0)
f = np.exp(-(grid.r.T**2))  # a radially symmetric Gaussian, f(r) = exp(-r^2)

signal = pypft.SpacePolarSignal(f, grid)
signal.domain


## Walking one edge at a time

`SpacePolarSignal` defines exactly one step method, `to_harmonics`, since
`SPACE_POLAR` has exactly one neighbour in the chain. Calling it returns a
`SpaceHarmonicSignal` -- a different, statically-known type:


In [ ]:
harmonic_signal = signal.to_harmonics()
type(harmonic_signal).__name__, harmonic_signal.domain


`SpaceHarmonicSignal` sits in the middle of the chain, so it has *two*
neighbours and two step methods: `to_angles` (back to `SPACE_POLAR`) and
`to_frequency` (onward to `FREQUENCY_HARMONIC`, through the discrete Hankel
transform). It has no `to_harmonics` of its own -- that method only exists
on `SpacePolarSignal`. Writing `harmonic_signal.to_harmonics()` by hand is
therefore rejected by `pyright` before the code ever runs, and raises
`AttributeError` if it somehow did run anyway:


In [ ]:
try:
    harmonic_signal.to_harmonics()  # type: ignore[attr-defined]
except AttributeError as exc:
    print(exc)


## Composing the full chain

Chaining every step method by hand reaches `FREQUENCY_POLAR` -- and
matches `forward_pft` on the same array exactly, since each step method is
a thin wrapper around the same `pypft.dft`/`pypft.transform` calls
`forward_pft` itself makes:


In [ ]:
by_hand = signal.to_harmonics().to_frequency().to_angles()

expected = pypft.forward_pft(f, grid)
float(np.abs(by_hand.values - expected).max())


For a target domain further away than one call away -- or simply not known
until runtime -- `to` walks the same chain dynamically, choosing the right
step methods and direction on its own:


In [ ]:
walked = signal.to(pypft.Domain.FREQUENCY_POLAR)

walked.domain, float(np.abs(walked.values - expected).max())


`to` still only ever moves one verified edge at a time internally; it
just picks the edges for you. Asking it for a domain that is not one of
the four `Domain` members raises immediately, the same way any other
PyPFT entry point rejects an invalid argument:


In [ ]:
try:
    signal.to("FREQUENCY_POLAR")  # not a Domain member
except TypeError as exc:
    print(exc)


## Where to go next

`pypft.domains` is deliberately thin: `forward_pft`/`inverse_pft` remain
the array-in, array-out primitive, and nothing in `pypft.transform` or
`pypft.grid` requires wrapping a signal in one of these classes at all.
Later notebooks cover 3-D batches and performance, then visualization.
